__Exploratory Data Analysis (EDA)__

- Inspect distributions, missing values, outliers.
- Plot time series of IV, skew, curvature.
- Compare SPY vs QQQ.
- Correlation checks.
- Document findings.

In [1]:
# import parquet_extractor  
# import importlib

# importlib.reload(parquet_extractor) 

In [2]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from parquet_extractor import load_ticker_year_data, get_ticker_metadata

__Load the filtered parquet__

In [3]:
# metadata = get_ticker_metadata('SPY', Path("."))
# metadata

In [4]:
def load_merged_ticker_data(tickers, start_year, end_year, data_dir=Path(".")):
    """Load and merge data for multiple tickers across a range of years."""
    all_data = []

    if isinstance(tickers, str):
        tickers = [tickers]

    for ticker in tickers:
        for year in range(start_year, end_year + 1):
            try:
                df = load_ticker_year_data(ticker, year, data_dir)
                df = df.with_columns([
                    pl.lit(ticker).alias("ticker"),
                ])
                all_data.append(df)
            except FileNotFoundError:
                print(f"Data not found for {ticker} in {year}, skipping.")
            except Exception as e:
                print(f"Error loading {ticker} in {year}: {e}")

    if all_data:
        return pl.concat(all_data)
    else:
        return pl.DataFrame()

In [5]:
spy_pl = load_merged_ticker_data('SPY', 2022, 2023, Path("."))
spy_df = spy_pl.to_pandas()
spy_df.sample(7)

,date,secid,symbol,cp_flag,exdate,strike_price,best_bid,best_offer,volume,open_interest,...,price_diff_5d,price_diff_8d,price_diff_13d,price_diff_21d,price_diff_34d,price_diff_55d,price_diff_89d,price_diff_144d,price_diff_233d,ticker
2901886,2022-03-18,109820.0,SPY 230317C505000,C,2023-03-17,505000.0,9.25,11.37,15.0,2377.0,...,0.00000,3.44998,3.44998,18.34998,27.51999,19.03998,25.09000,22.56998,-8.43002,SPY
1686418,2022-06-28,109820.0,SPY 221216C414000,C,2022-12-16,414000.0,11.02,11.09,10.0,2675.0,...,0.00000,-7.94001,-9.43000,2.58999,5.57998,6.77999,-9.15000,-36.74002,-27.67002,SPY
4617708,2023-05-09,109820.0,SPY 241220C480000,C,2024-12-20,480000.0,13.55,16.04,30.0,3683.0,...,5.79999,5.79999,-4.58002,-5.00000,6.57000,-3.21002,2.88000,17.75998,12.66000,SPY
3462879,2023-04-21,109820.0,SPY 230721P360000,P,2023-07-21,360000.0,2.86,2.87,1781.0,23127.0,...,0.00000,0.32001,0.32001,-1.94000,-2.00998,-0.25998,4.15002,1.25000,19.03000,SPY
498768,2022-12-12,109820.0,SPY 230616P330000,P,2023-06-16,330000.0,6.53,6.79,114.0,21417.0,...,0.00000,5.67001,5.67001,2.71002,5.79001,-0.63999,-8.72998,4.36001,19.00000,SPY
2102614,2022-08-25,109820.0,SPY 221216C435000,C,2022-12-16,435000.0,11.99,12.14,23.0,7186.0,...,0.00000,0.00000,5.84000,7.16000,6.16000,-2.63000,-10.19000,-0.47998,7.52002,SPY
3113527,2023-07-13,109820.0,SPY 230915P210000,P,2023-09-15,210000.0,0.03,0.04,1.0,3439.0,...,3.54001,3.54001,7.10001,7.10001,11.45001,18.12000,12.38001,31.70999,36.70999,SPY


In [6]:
qqq_pl = load_merged_ticker_data('QQQ', 2022, 2023, Path("."))
qqq_df = qqq_pl.to_pandas()
qqq_df.sample(7)

,date,secid,symbol,cp_flag,exdate,strike_price,best_bid,best_offer,volume,open_interest,...,price_diff_5d,price_diff_8d,price_diff_13d,price_diff_21d,price_diff_34d,price_diff_55d,price_diff_89d,price_diff_144d,price_diff_233d,ticker
98005,2022-10-18,107899.0,QQQ 230120P225000,P,2023-01-20,225000.0,4.45,4.48,215.0,7519.0,...,2.13000,2.13000,2.13000,10.74002,2.66000,5.07001,-2.04999,-4.03000,-27.91998,QQQ
634658,2022-05-04,107899.0,QQQ 220630C336000,C,2022-06-30,336000.0,11.27,11.45,2.0,111.0,...,10.78000,10.78000,11.12000,1.59000,12.46000,-4.54998,-9.08999,-30.04998,6.25000,QQQ
854832,2022-04-05,107899.0,QQQ 220414P375000,P,2022-04-14,375000.0,15.04,15.24,355.0,2910.0,...,-8.19998,-8.19998,-0.75000,-1.44000,-3.80999,1.45002,9.61002,29.83002,20.61002,QQQ
136773,2022-09-13,107899.0,QQQ 220930P245000,P,2022-09-30,245000.0,0.40,0.42,131.0,2669.0,...,-17.03998,-13.38999,-6.81998,-5.26999,-5.69998,-13.73999,-38.57999,-29.19000,4.66000,QQQ
1334972,2023-04-27,107899.0,QQQ 230630C314000,C,2023-06-30,314000.0,16.24,16.42,52.0,443.0,...,0.00000,8.48001,8.48001,10.36002,4.40000,4.07001,1.18000,-0.57998,21.42002,QQQ
46976,2022-05-09,107899.0,QQQ 220520P200000,P,2022-05-20,200000.0,0.04,0.06,249.0,1751.0,...,-12.10001,-15.85001,-15.85001,-32.45002,-21.33002,-19.61002,-49.11002,-72.15000,-27.25000,QQQ
602066,2022-03-10,107899.0,QQQ 220318C332000,C,2022-03-18,332000.0,6.74,6.79,2322.0,2207.0,...,0.00000,0.00000,-3.72000,7.91998,-6.03000,-10.22000,-9.22000,-24.77002,-22.28000,QQQ


In [7]:
# Save to Parquet
spy_pl.write_parquet("./parquet/spy_historical_2022_2023.parquet")
qqq_pl.write_parquet("./parquet/qqq_historical_2022_2023.parquet")

In [8]:
spl = pl.read_parquet("./parquet/spy_historical_2022_2023.parquet")
qpl = pl.read_parquet("./parquet/qqq_historical_2022_2023.parquet")
print(spl.shape)
print(qpl.shape)

(4730700, 37)
(1722846, 37)
